In [3]:
import os
from pypdf import PdfReader

def load_pdfs(folder_path):
    texts = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            reader = PdfReader(os.path.join(folder_path, file))
            
            text = ""
            for page in reader.pages:
                text += page.extract_text() + "\n"
            
            texts.append(text)
    
    return texts


def load_txt(folder_path):
    texts = []
    
    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            with open(os.path.join(folder_path, file), "r", encoding="utf-8") as f:
                texts.append(f.read())
    
    return texts


# Load everything
pdf_texts = load_pdfs("data/papers")
note_texts = load_txt("data/notes")

all_docs = pdf_texts + note_texts

print(f"Loaded {len(all_docs)} documents")

Loaded 9 documents


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

all_chunks = []

for doc in all_docs:
    chunks = splitter.split_text(doc)
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")

c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Total chunks: 1037


In [5]:
def clean_chunks(chunks):
    cleaned = []
    
    for c in chunks:
        if len(c.split()) < 20:
            continue  # remove tiny chunks
        
        if "copyright" in c.lower():
            continue
        
        cleaned.append(c)
    
    return cleaned

all_chunks = clean_chunks(all_chunks)

print("Cleaned chunks:", len(all_chunks))

Cleaned chunks: 1030


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=2
)

embeddings = vectorizer.fit_transform(all_chunks).toarray()

In [7]:
import numpy as np

def search_docs(query, k=5):
    query_vec = vectorizer.transform([query]).toarray()
    
    scores = np.dot(embeddings, query_vec.T).flatten()
    
    k = min(k, len(all_chunks))  # safety fix
    
    top_k_idx = np.argsort(scores)[-k:][::-1]
    
    results = []
    for i in top_k_idx:
        if i < len(all_chunks):
            results.append(all_chunks[i])
    
    return results

In [8]:
query = "YLD duration estimation without follow up survival data"
results = search_docs(query, k=5)

for i, r in enumerate(results):
    print(f"\nResult {i+1}:\n", r[:300])


Result 1:
 What We Do NOT Have
Duration per phase → Not available



What Do We Need
Survival Time to derive duration.
How is survival time linked to the duration phase?
If so, what data is required to estimate survival time?
Do we have the data? If not can we collect or obtain?
Are there other methods to deri

Result 2:
 Disease Burden Estimation
YLD (Years Lived with Disability)
YLD Formula
YLD = I × DW × L
I is the number of incident cases in the reference period 
DW is the disability weight (in the range 0–1)
L is the average duration of disability (measured in years).

Result 3:
 throughout the country.
8 India is committed to meeting
the WHO ’s target of eliminating cervical cancer. 9 Pop-
ulation based survival data are used to assess the ef ﬁ-
ciency and effectiveness of cancer diagnostic, treatment
and follow-up services in the region. In 2017, the study
on Population Ba

Result 4:
 and the DOD (any cause of death) or loss to follow-up or
censoring is used to compute survival

In [ ]:
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [10]:
def research_agent(query, retrieved_chunks):
    
    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    You are an expert in disease burden estimation.

    Use ONLY the context below to answer the question.

    Context:
    {context}

    Question:
    {query}

    Instructions:
    - Be precise
    - Combine information
    - If something is missing, say assumptions clearly
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )

    return response.choices[0].message.content

In [11]:
def search_agent(query, k=5):
    
    # Step 1: retrieve chunks from vector search
    retrieved_chunks = search_docs(query, k=k)
    
    # Step 2: basic filtering (remove empty/noisy chunks)
    cleaned_chunks = []
    
    for chunk in retrieved_chunks:
        if chunk is None:
            continue
        
        if len(chunk.strip()) < 50:
            continue
        
        cleaned_chunks.append(chunk)
    
    # Step 3: print retrieved chunks (for debugging)
    print("\n🔎 Retrieved Context:\n")
    
    for i, chunk in enumerate(cleaned_chunks):
        print(f"Chunk {i+1}:")
        print(chunk[:300])
        print("------")
    
    return cleaned_chunks

In [12]:
query = "How to estimate YLD when follow-up time is not available?"

retrieved = search_agent(query)

answer = research_agent(query, retrieved)

print("\n🧠 Final Answer:\n", answer)


🔎 Retrieved Context:

Chunk 1:
melanoma of skin and prostate), smaller age groups were 
used: < 50, 50–64, 65–69, 70–74, 75–79, 80–84, and 85+. 
If the age group specific estimates were not available due 
to too few cases, we used the all-ages estimate instead. 
Likewise, if the region-specific estimate was not available, 
we use
------
Chunk 2:
What We Do NOT Have
Duration per phase → Not available



What Do We Need
Survival Time to derive duration.
How is survival time linked to the duration phase?
If so, what data is required to estimate survival time?
Do we have the data? If not can we collect or obtain?
Are there other methods to deri
------
Chunk 3:
sis, since no patients in that selection had a follow-up of 
minimum 9/10 years, so the estimate at the time point(s) 
does not exist. In these cases, we propagated the last 
observed (annual) survival probabilities to the years with-
out estimates using the interval specific survival (e.g., if 
no 
------
Chunk 4:
Step-wise Phase Y

In [13]:
def calculate_yld(incidence, duration, disability_weight):
    return incidence * duration * disability_weight

def estimate_duration(method="literature"):
    if method == "literature":
        return "Use average duration from published studies"
    elif method == "survival":
        return "Estimate duration using survival analysis"
    else:
        return "No method available"

In [14]:
def calculate_yll(deaths, life_expectancy):
    return deaths * life_expectancy

In [15]:
TOOLS = {
    "calculate_yld": calculate_yld,
    "estimate_duration": estimate_duration,
    "calculate_yll": calculate_yll
}

In [16]:
def tool_agent(query):
    
    if "calculate yld" in query.lower():
        return {
            "tool": "calculate_yld",
            "inputs": {
                "incidence": 1000,
                "duration": 2,
                "disability_weight": 0.3
            }
        }
    
    elif "duration" in query.lower():
        return {
            "tool": "estimate_duration",
            "inputs": {
                "method": "literature"
            }
        }
    
    else:
        return None

In [17]:
def run_tool(tool_call):
    if tool_call is None:
        return None
    
    tool_name = tool_call["tool"]
    inputs = tool_call["inputs"]
    
    return TOOLS[tool_name](**inputs)

In [18]:
def orchestrator(query):
    
    # Step 1: Retrieve
    docs = search_agent(query)
    
    # Step 2: Tool decision
    tool_call = tool_agent(query)
    tool_output = run_tool(tool_call)
    
    # Step 3: LLM reasoning
    context = "\n".join(docs)
    
    prompt = f"""
    Context:
    {context}
    
    Tool Output:
    {tool_output}
    
    Question:
    {query}
    
    Answer clearly.
    """
    
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )
    
    return response.choices[0].message.content

In [19]:
print(orchestrator("Calculate YLD for incidence 1000, duration 2, DW 0.3"))


🔎 Retrieved Context:

Chunk 1:
Disease Burden Estimation
YLD (Years Lived with Disability)
YLD Formula
YLD = I × DW × L
I is the number of incident cases in the reference period 
DW is the disability weight (in the range 0–1)
L is the average duration of disability (measured in years).
------
Chunk 2:
cific treatment or surgery-induced complications for 
the entire duration of illness. These complications com -
prised mastectomy (breast cancer; DW = 0.036), stoma 
(colorectal cancer; DW = 0.095), laryngectomy (larynx 
cancer; DW = 0.051), incontinence (prostate and bladder 
cancer; DW = 0.139), a
------
Chunk 3:
DW is the disability weight (in the range 0 –1),
L is the average duration of disability (measured in years).
Prevalence approach:
YLD ¼ P/C3 DW /C3 S
Where:
P is the number of prevalent cases in the reference period,
DW is the disability weight (in the range 0 –1),
S is the severity proportion.
von
------
Chunk 4:
multiplying the prevalence or incidence of a disease by
the sh

In [20]:
chat_history = []

In [21]:
def orchestrator(query):
    
    # Step 1: Retrieve
    docs = search_agent(query)
    
    # Step 2: Tool usage
    tool_call = tool_agent(query)
    tool_output = run_tool(tool_call)
    
    # Step 3: Add history
    history_text = "\n".join(chat_history[-5:])  # last 5 messages
    
    # Step 4: LLM
    prompt = f"""
    You are a Disease Burden Estimation assistant.

    Chat History:
    {history_text}

    Context:
    {docs}

    Tool Output:
    {tool_output}

    Question:
    {query}

    Answer clearly and conversationally.
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    answer = response.choices[0].message.content
    
    # Step 5: Save history
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {answer}")
    
    return answer

In [22]:
print(orchestrator("What is YLD?"))
print(orchestrator("How do we estimate it without follow-up?"))


🔎 Retrieved Context:

Chunk 1:
Step-wise Phase YLD Calculation
1️⃣ Diagnosis Phase
YLD=1000×0.5×0.30=150
2️⃣ Controlled Phase
YLD=800×3×0.10=240
3️⃣ Metastatic Phase
YLD=500×1×0.45=225
4️⃣ Terminal Phase
YLD=100×0.5×0.60=30
Total YLD
Total YLD = 150 + 240+ 225 + 30 = 645
So,
1,000 incident cancer cases result in 645 Years Lived w
------
Chunk 2:
mates also identified colorectal, breast and lung cancer 
as accounting for the most YLD. In particular colorec -
tal cancer accounted for 16% of all YLD due to cancer in 
Spain in 2000 [3]. Breast, colorectal and prostate resulted 
also to have the highest number of age-standardized YLD 
rate in It
------
Chunk 3:
Disease Burden Estimation
YLD (Years Lived with Disability)
YLD Formula
YLD = I × DW × L
I is the number of incident cases in the reference period 
DW is the disability weight (in the range 0–1)
L is the average duration of disability (measured in years).
------
Chunk 4:
(87 DALYs per 100,000 population, 33 % of YLD s) for breast ca

In [23]:
pip install google-search-results

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
from serpapi import GoogleSearch

def web_search(query):
    params = {
        "q": query,
        "api_key": "YOUR_SERPER_API_KEY"
    }
    
    search = GoogleSearch(params)
    results = search.get_dict()
    
    snippets = []
    for r in results.get("organic_results", [])[:3]:
        snippets.append(r.get("snippet", ""))
    
    return "\n".join(snippets)

In [25]:
TOOLS["web_search"] = web_search

In [26]:
def tool_agent(query):
    
    if "latest" in query.lower() or "recent" in query.lower():
        return {
            "tool": "web_search",
            "inputs": {"query": query}
        }
    
    elif "calculate yld" in query.lower():
        return {
            "tool": "calculate_yld",
            "inputs": {
                "incidence": 1000,
                "duration": 2,
                "disability_weight": 0.3
            }
        }
    
    return None

In [27]:
def calculate_yld(incidence, duration, disability_weight):
    return incidence * duration * disability_weight


def calculate_yll(deaths, life_expectancy):
    return deaths * life_expectancy


def estimate_duration():
    return "Use survival estimates or literature-based duration"

In [28]:
import requests

def web_search(query):
    url = f"https://api.duckduckgo.com/?q={query}&format=json"
    response = requests.get(url).json()
    
    results = []
    
    if "RelatedTopics" in response:
        for r in response["RelatedTopics"][:3]:
            if "Text" in r:
                results.append(r["Text"])
    
    return "\n".join(results)

In [29]:
TOOLS = {
    "calculate_yld": calculate_yld,
    "calculate_yll": calculate_yll,
    "estimate_duration": estimate_duration,
    "web_search": web_search
}

In [30]:
def tool_agent(query):
    q = query.lower()
    
    if "calculate yld" in q:
        return {
            "tool": "calculate_yld",
            "inputs": {
                "incidence": 1000,
                "duration": 2,
                "disability_weight": 0.3
            }
        }
    
    elif "calculate yll" in q:
        return {
            "tool": "calculate_yll",
            "inputs": {
                "deaths": 500,
                "life_expectancy": 20
            }
        }
    
    elif "duration" in q:
        return {
            "tool": "estimate_duration",
            "inputs": {}
        }
    
    elif "latest" in q or "recent" in q:
        return {
            "tool": "web_search",
            "inputs": {"query": query}
        }
    
    return None

In [44]:
def search_agent(query):

    docs, scores = search_docs(query, k=5)

    avg_score = scores.mean()

    print("Retrieval confidence:", avg_score)

    return docs, avg_score

In [35]:
def orchestrator(query):
    
    # Step 1: Retrieve context
    docs = search_agent(query)
    context = "\n\n".join(docs)
    
    # Step 2: Tool decision
    tool_call = tool_agent(query)
    tool_output = run_tool(tool_call)
    
    # Step 3: Chat memory
    history = "\n".join(chat_history[-6:])
    
    # Step 4: LLM reasoning
    prompt = f"""
    You are an expert in Disease Burden Estimation.

    Chat History:
    {history}

    Context:
    {context}

    Tool Output:
    {tool_output}

    Question:
    {query}

    Instructions:
    - Use tool output if available
    - Use context for explanation
    - Be clear and structured

    Format:
    1. Answer
    2. Explanation
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    answer = response.choices[0].message.content
    
    # Save history
    chat_history.append(f"User: {query}")
    chat_history.append(f"Assistant: {answer}")
    
    return answer, context

In [33]:
while True:
    query = input("\nYou: ")
    
    if query.lower() in ["exit", "quit"]:

        break
    
    response = orchestrator(query)
    print("\nAssistant:\n", response)


Assistant:
 **1. Answer:**

The latest methods for disease burden estimation include:

1. **Disability-Adjusted Life Years (DALYs)**: A measure of overall disease burden, expressed as the number of years lost due to ill-health, disability, or early death.
2. **Years Lived with Disability (YLD)**: A measure of the number of years that people live with a disability due to a particular disease or condition.
3. **Years of Life Lost (YLL)**: A measure of the number of years lost due to premature death.
4. **Global Burden of Disease (GBD)**: A comprehensive framework for measuring and comparing the burden of diseases and injuries across different populations and regions.
5. **Machine Learning and Artificial Intelligence (ML/AI)**: Techniques used to analyze large datasets and identify patterns and trends in disease burden estimation.

**2. Explanation:**

The latest methods for disease burden estimation have evolved to incorporate new data sources, advanced statistical techniques, and a mor

In [34]:
def evaluate_rag(query, answer, context):

    prompt = f"""
You are evaluating a RAG system.

Query:
{query}

Answer:
{answer}

Context used:
{context}

Score the following from 1 to 5:

1. Context relevance – retrieved context relevant to query?
2. Faithfulness – answer supported by context?
3. Answer quality – clear and useful answer?

Return format:
Context relevance: X
Faithfulness: X
Answer quality: X
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [40]:
query = "How to estimate YLD duration when follow-up survival data is missing"

answer, context = orchestrator(query)

scores = evaluate_rag(query, answer, context)

print(answer)
print("\nEvaluation:\n", scores)

**1. Answer:**

To estimate YLD duration when follow-up survival data is missing, we can use the tool output to estimate the survival time and then use it to derive the duration. Alternatively, we can use literature-based duration or survival estimates.

**2. Explanation:**

When follow-up survival data is missing, we can use the tool output to estimate the survival time. The tool output suggests using survival estimates or literature-based duration. This is because the survival time is directly linked to the duration of each phase, and we can use the survival estimates to estimate the duration.

If we don't have the tool output, we can use literature-based duration or survival estimates. This involves using published studies or literature to estimate the survival time and duration of each phase.

In the context of cervical cancer, the study on Population Based Cancer Survival (PBCS) on breast, cervical, and head and neck cancers in India was launched across 25 PBCR under NCRP. However

In [41]:
results = search_docs("YLD prevalence formula", k=5)

for r in results:
    print(r[:300])

Step-wise Phase YLD Calculation
1️⃣ Diagnosis Phase
YLD=1000×0.5×0.30=150
2️⃣ Controlled Phase
YLD=800×3×0.10=240
3️⃣ Metastatic Phase
YLD=500×1×0.45=225
4️⃣ Terminal Phase
YLD=100×0.5×0.60=30
Total YLD
Total YLD = 150 + 240+ 225 + 30 = 645
So,
1,000 incident cancer cases result in 645 Years Lived w
Disease Burden Estimation
YLD (Years Lived with Disability)
YLD Formula
YLD = I × DW × L
I is the number of incident cases in the reference period 
DW is the disability weight (in the range 0–1)
L is the average duration of disability (measured in years).
standardized prevalence rates from 3581 to 3770 per 
100,000 can also be seen over the same period (5%). A 
similar trend was observed for the total number of cancer 
associated prevalence-based YLD, with an increase from 
45,887 to 51,464 YLD (12%), reflected in an increase in 
the age-standardized
prevalence as the original population subgroup of
interest, diseases are independently assigned to
simulants by assuming disease prevalence to

In [42]:
def rewrite_query(query):

    prompt = f"""
Rewrite the query to improve document retrieval.

Query: {query}

Return a clearer scientific search query.
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [43]:
def search_docs(query, k=5):

    query_vec = vectorizer.transform([query]).toarray()

    scores = np.dot(embeddings, query_vec.T).flatten()

    top_idx = np.argsort(scores)[-k:][::-1]

    results = [all_chunks[i] for i in top_idx]

    return results, scores[top_idx]

In [51]:
def orchestrator(query):

    docs, score = search_agent(query)

    # fallback to web search
    if score < 0.05:
        print("⚠️ Low retrieval confidence → using web search")
        web_context = web_search(query)
        context = web_context
    else:
        context = "\n".join(docs)

    tool_call = tool_agent(query)
    tool_output = run_tool(tool_call)

    prompt = f"""
Context:
{context}

Tool Output:
{tool_output}

Question:
{query}

Answer clearly.
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}],
        temperature=0.2
    )

    return response.choices[0].message.content

In [52]:
def evaluate_rag(query, answer, context):

    prompt = f"""
Evaluate this RAG system response.

Query:
{query}

Answer:
{answer}

Context:
{context}

Score from 1–5:

1. Context relevance
2. Faithfulness to context
3. Answer quality

Return format:

Context relevance: X
Faithfulness: X
Answer quality: X
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [53]:
query = "What is the YLD formula using prevalence?"

answer = orchestrator(query)

scores = evaluate_rag(query, answer, context)

print(answer)
print(scores)

Retrieval confidence: 0.23080484140330487
The YLD formula using prevalence is:

burden for individual = 1 - (1 - disability weightd) / C0/C1

However, the simplified YLD formula using prevalence is:

burden for individual = 1 - (1 - disability weightd) 

But the more commonly used YLD formula is:

YLD = I × DW × L

Where:
- I is the number of incident cases in the reference period
- DW is the disability weight (in the range 0–1)
- L is the average duration of disability (measured in years)

But if we are using prevalence, then the formula is:

YLD = P × DW × L

Where:
- P is the prevalence of the disease
- DW is the disability weight (in the range 0–1)
- L is the average duration of disability (measured in years)
Based on the provided RAG system response, I would evaluate it as follows:

**Context Relevance: 5**
The response is highly relevant to the context. It addresses the specific question about the YLD formula using prevalence and provides multiple formulas. Additionally, it discu

In [54]:
query = rewrite_query(query)
docs, score = search_agent(query)

Retrieval confidence: 0.20150546960275412


In [55]:
if score < 0.05:
    context = web_search(query)

In [61]:
def evaluate_rag(query, answer, context):

    prompt = f"""
Evaluate this RAG response.

Query:
{query}

Answer:
{answer}

Context:
{context}

Score from 1 to 5:

1. Context relevance – retrieved context relevant?
2. Faithfulness – answer supported by context?
3. Answer quality – clear and useful answer?
4. Context utilization – did answer use the context?
5. Hallucination risk – unsupported claims?
6. Tool correctness – correct use of tools/calculations?

Return format:

Context relevance: X
Faithfulness: X
Answer quality: X
Context utilization: X
Hallucination risk: X
Tool correctness: X
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    return response.choices[0].message.content

In [73]:
def precision_at_k(relevant_docs, retrieved_docs, k):

    retrieved_k = retrieved_docs[:k]

    hits = 0

    for doc in retrieved_k:
        doc_text = str(doc).lower()

        for rel in relevant_docs:
            if rel.lower() in doc_text:
                hits += 1
                break

    precision = hits / k

    return precision

In [74]:
def recall_at_k(relevant_docs, retrieved_docs, k):

    retrieved_k = retrieved_docs[:k]

    hits = 0

    for rel in relevant_docs:
        rel_text = rel.lower()

        for doc in retrieved_k:
            if rel_text in str(doc).lower():
                hits += 1
                break

    recall = hits / len(relevant_docs)

    return recall

In [64]:
def retrieval_confidence(scores):

    return scores.mean()

In [65]:
query = "What is the YLD formula using prevalence?"

answer = orchestrator(query)

scores = evaluate_rag(query, answer, context)

print(answer)
print(scores)

Retrieval confidence: 0.23080484140330487
The YLD formula using prevalence is:

burden for individual = 1 - (1 - disability weightd) / C0/C1

However, the simplified YLD formula is:

YLD = I × DW × L

Where:
- I is the number of incident cases in the reference period
- DW is the disability weight (in the range 0–1)
- L is the average duration of disability (measured in years)

Note that the formula using prevalence is more complex and is used for estimating disease burden in a simulated population, while the simplified formula is used for calculating YLD from incident cases.
Here's the evaluation of the RAG response:

1. **Context relevance: 5**
The context is highly relevant to the question asked. The response starts by addressing the question directly and then delves into the limitations of the data, which is a crucial aspect of the context.

2. **Faithfulness: 5**
The answer is entirely faithful to the context. The response acknowledges the limitations of the data and explains how i

In [69]:
queries = [
    "What is YLD formula?",
    "What is prevalence YLD?",
    "How is YLL calculated?"
]

In [71]:
ground_truth = {
    
    "What is YLD formula?": [
        "YLD = I × DW × L",
        "YLD = Incidence × Disability Weight × Duration"
    ],

    "What is prevalence YLD?": [
        "YLD = Prevalence × Disability Weight",
        "Prevalence approach: YLD = P × DW"
    ],

    "How is YLL calculated?": [
        "YLL = Number of deaths × Life expectancy",
        "Years of Life Lost is calculated as deaths multiplied by standard life expectancy"
    ]
}

In [75]:
k = 5

precision_scores = []
recall_scores = []

for query in queries:

    retrieved_docs = search_docs(query, k=k)

    relevant_docs = ground_truth[query]

    p = precision_at_k(relevant_docs, retrieved_docs, k)
    r = recall_at_k(relevant_docs, retrieved_docs, k)

    precision_scores.append(p)
    recall_scores.append(r)

    print("Query:", query)
    print("Precision@", k, ":", p)
    print("Recall@", k, ":", r)
    print("-"*40)

Query: What is YLD formula?
Precision@ 5 : 0.2
Recall@ 5 : 0.5
----------------------------------------
Query: What is prevalence YLD?
Precision@ 5 : 0.0
Recall@ 5 : 0.0
----------------------------------------
Query: How is YLL calculated?
Precision@ 5 : 0.0
Recall@ 5 : 0.0
----------------------------------------


In [76]:
avg_precision = sum(precision_scores) / len(precision_scores)
avg_recall = sum(recall_scores) / len(recall_scores)

print("Average Precision@", k, ":", avg_precision)
print("Average Recall@", k, ":", avg_recall)

Average Precision@ 5 : 0.06666666666666667
Average Recall@ 5 : 0.16666666666666666
